<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z318_HAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HAR  (Heterogeneous AutoRegressive)

Un modelo por producto. Para cada `product_id` se ajusta una regresion lineal con features construidas a partir de promedios de la misma serie a distintos horizontes:

```
tn_t = c + β1·tn_{t-1} + β3·mean(tn_{t-3:t-1}) + β6·mean(tn_{t-6:t-1}) + β12·mean(tn_{t-12:t-1}) + ε
```

**Por que no VAR**: con 780 productos un VAR completo necesitaria estimar 780² parametros por lag con solo 36 observaciones — inviable. El HAR por producto es la alternativa practica.

La prediccion de `202002` requiere dos pasos:
1. predecir `202001` usando los ultimos 12 meses conocidos
2. predecir `202002` usando `202001` (predicho) + los 11 meses anteriores conocidos

## 0.1 Init ambiente Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Modelo HAR

## 1.1 Init Experimento

In [ ]:
!pip install uv
!uv pip install -q kaggle

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
  import os
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  os.system(comando)

In [ ]:
import os
import numpy as np
import polars as pl
from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings('ignore')

Por favor, cargar aqui SU semilla primigenia

In [ ]:
PARAM = {
  'experimento': 'HAR-01',
  'kaggle_competition': 'labo-iii-2026-rosario',
  'semilla_primigenia': 102191
}

In [ ]:
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

## 1.2 Preparacion de datos

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

In [ ]:
tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

In [ ]:
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")

print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir, on="product_id", how="inner")
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

display(tb_ventas)

## 1.3 Construccion de features HAR

Para cada fila `t` se construyen:
- `lag1`  : `tn_{t-1}`
- `lag3`  : `mean(tn_{t-3}, tn_{t-2}, tn_{t-1})`
- `lag6`  : `mean(tn_{t-6}, ..., tn_{t-1})`
- `lag12` : `mean(tn_{t-12}, ..., tn_{t-1})`

Se necesitan al menos 12 observaciones pasadas para construir todas las features, por lo tanto el entrenamiento arranca desde la observacion 13 en adelante.

In [ ]:
def build_har_features(serie: np.ndarray):
    """
    Dado un vector de T valores, devuelve (X, y) para regresion HAR.
    Cada fila de X corresponde a un tiempo t >= 12:
      [lag1, mean_lag3, mean_lag6, mean_lag12]
    y[i] = serie[t]
    """
    T = len(serie)
    rows_X, rows_y = [], []

    for t in range(12, T):
        lag1   = serie[t - 1]
        mean3  = serie[t-3 : t].mean()
        mean6  = serie[t-6 : t].mean()
        mean12 = serie[t-12: t].mean()
        rows_X.append([lag1, mean3, mean6, mean12])
        rows_y.append(serie[t])

    return np.array(rows_X), np.array(rows_y)


def har_predict_next(modelo, serie: np.ndarray) -> float:
    """
    Predice el siguiente valor (t+1) a partir del final de la serie conocida.
    """
    t = len(serie)
    lag1   = serie[t - 1]
    mean3  = serie[t-3 : t].mean()
    mean6  = serie[t-6 : t].mean()
    mean12 = serie[t-12: t].mean()
    X_pred = np.array([[lag1, mean3, mean6, mean12]])
    return float(modelo.predict(X_pred)[0])

## 1.4 Ajuste HAR por producto y prediccion a dos pasos

In [ ]:
productos = tb_apredecir["product_id"].to_list()
resultados = []

for producto in productos:
    print(producto, end=' ')

    serie = (
        tb_ventas
        .filter(pl.col("product_id") == producto)
        .sort("periodo")["tn"]
        .to_numpy()
        .astype(float)
    )

    try:
        X, y = build_har_features(serie)

        modelo = LinearRegression()
        modelo.fit(X, y)

        # paso 1: predecir 202001
        pred_1 = har_predict_next(modelo, serie)
        pred_1 = max(pred_1, 0.0)

        # paso 2: predecir 202002 usando la serie extendida con pred_1
        serie_ext = np.append(serie, pred_1)
        pred_2 = har_predict_next(modelo, serie_ext)
        pred_2 = max(pred_2, 0.0)

    except Exception as e:
        print(f"  ERROR {producto}: {e}")
        # fallback: promedio de los ultimos 12 meses
        pred_2 = float(serie[-12:].mean()) if len(serie) >= 12 else float(serie.mean())

    resultados.append({'product_id': producto, 'tn': pred_2})

print("\nlisto")

## 1.5 Armado del submit

In [ ]:
tb_final = pl.DataFrame(resultados)
display(tb_final)
print("nulls:", tb_final["tn"].is_null().sum())

## 1.6 Submit a Kaggle

In [ ]:
archivo = "HAR.csv"
mensaje = "HAR lag1-3-6-12 dos pasos"

tb_final.write_csv(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje)